# MSc in AI
## Lab Project: DIP & CV Exercises on a Custom Image Dataset

**Prepared for:** Dr. Natnael Argaw Wondimu
**prepared by:** Zerihun Ayele Chala (GSR/0979/18)
**Programme:** Master's in Artificial Intelligence, Addis Ababa University  
**Environment:** Python 3 + OpenCV  

---
This notebook follows the **Suggested Exercises** from each chapter exactly:

| Chapter | Topic | Exercises |
|---------|-------|-----------|
| CV Ch 1 | Image I/O & Color Spaces | Load/display, save PNG/BMP, LAB/YCrCb, cv2.split() |
| DIP Ch 2 | Point Operations | Grayscale (luminosity), brightness slider fn, histogram eq per channel |
| DIP Ch 3 | Segmentation | Otsu on all images, Sobel/Prewitt/Canny compare, region-growing |
| DIP Ch 4 | Morphology | Different SEs, opening/closing on noisy image, morph text extraction |
| DIP Ch 5 | Filtering | Filter size study, unsharp masking, PSNR/SSIM, patch correlation |

## Setup — Imports and Shared Utilities

In [1]:
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

IMAGE_DIR  = 'images'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_NAMES = [
    'traffic_light', 'person', 'natural_scene', 'road',
    'simple_object', 'pepper', 'fruits', 'flowers',
    'parrot', 'low_contrast', 'night_scene'
]

def load(name):
    img = cv2.imread(os.path.join(IMAGE_DIR, f'{name}.jpeg'))
    if img is None:
        raise FileNotFoundError(name)
    return img

def shrink(img, max_side=600):
    h, w = img.shape[:2]
    s = min(max_side / max(h, w), 1.0)
    return cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA) if s < 1 else img

def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def savefig(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=110, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved → {name}')

print('OpenCV', cv2.__version__, '— setup complete.')

OpenCV 4.13.0 — setup complete.


---
## Chapter 1 — Image I/O and Color Spaces

**Suggested Exercises:**
1. Load and display your own image.
2. Save the image in PNG and BMP formats.
3. Try converting the image to other color spaces (e.g., LAB, YCrCb).
4. Use `cv2.split()` to view individual RGB/H channels.


### Exercise 1.1 — Load and display all 11 images

In [2]:
cols = 4
rows = -(-len(IMAGE_NAMES) // cols)   # ceiling division
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
fig.suptitle('Exercise 1.1 — Our Image Dataset (11 images)', fontsize=14, fontweight='bold')
axes = axes.ravel()

for i, name in enumerate(IMAGE_NAMES):
    img = shrink(load(name))
    axes[i].imshow(rgb(img))
    axes[i].set_title(name.replace('_', ' ').title(), fontsize=9)
    axes[i].axis('off')

for j in range(len(IMAGE_NAMES), len(axes)):
    axes[j].axis('off')

savefig(fig, 'ex1_1_load_display_all.png')

Saved → ex1_1_load_display_all.png


### Exercise 1.2 — Save each image in PNG and BMP formats

In [3]:
png_dir = os.path.join(OUTPUT_DIR, 'ex1_2_png')
bmp_dir = os.path.join(OUTPUT_DIR, 'ex1_2_bmp')
os.makedirs(png_dir, exist_ok=True)
os.makedirs(bmp_dir, exist_ok=True)

for name in IMAGE_NAMES:
    img = load(name)
    cv2.imwrite(os.path.join(png_dir, f'{name}.png'), img)
    cv2.imwrite(os.path.join(bmp_dir, f'{name}.bmp'), img)

print(f'Saved {len(IMAGE_NAMES)} images as PNG → {png_dir}')
print(f'Saved {len(IMAGE_NAMES)} images as BMP → {bmp_dir}')

# Verify by re-reading and checking shapes
sample_png = cv2.imread(os.path.join(png_dir, 'parrot.png'))
sample_bmp = cv2.imread(os.path.join(bmp_dir, 'parrot.bmp'))
print(f'\nVerification (parrot):')
print(f'  PNG shape: {sample_png.shape}')
print(f'  BMP shape: {sample_bmp.shape}')

Saved 11 images as PNG → outputs/ex1_2_png
Saved 11 images as BMP → outputs/ex1_2_bmp

Verification (parrot):
  PNG shape: (1606, 2048, 3)
  BMP shape: (1606, 2048, 3)


### Exercise 1.3 — Convert images to LAB and YCrCb color spaces

Beyond Grayscale and HSV (covered in lecture), we now explore:
- **LAB**: Perceptually uniform; L = lightness, A = green↔red, B = blue↔yellow
- **YCrCb**: Used in JPEG/video; Y = luma, Cr/Cb = colour difference channels


In [4]:
# Apply to all 11 images — show original, LAB, and YCrCb side by side
for name in IMAGE_NAMES:
    img   = shrink(load(name))
    lab   = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(f'Exercise 1.3 — Color Spaces: {name.replace("_"," ").title()}',
                 fontsize=11, fontweight='bold')
    axes[0].imshow(rgb(img));  axes[0].set_title('Original (BGR→RGB)'); axes[0].axis('off')
    axes[1].imshow(lab);       axes[1].set_title('LAB');                 axes[1].axis('off')
    axes[2].imshow(ycrcb);     axes[2].set_title('YCrCb');              axes[2].axis('off')
    savefig(fig, f'ex1_3_colorspaces_{name}.png')

Saved → ex1_3_colorspaces_traffic_light.png
Saved → ex1_3_colorspaces_person.png
Saved → ex1_3_colorspaces_natural_scene.png
Saved → ex1_3_colorspaces_road.png
Saved → ex1_3_colorspaces_simple_object.png
Saved → ex1_3_colorspaces_pepper.png
Saved → ex1_3_colorspaces_fruits.png
Saved → ex1_3_colorspaces_flowers.png
Saved → ex1_3_colorspaces_parrot.png
Saved → ex1_3_colorspaces_low_contrast.png
Saved → ex1_3_colorspaces_night_scene.png


### Exercise 1.4 — Use `cv2.split()` to view individual channels

We split both the **BGR** image and the **HSV** image to inspect each channel separately.


In [5]:
for name in IMAGE_NAMES:
    img = shrink(load(name))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Split BGR channels
    b, g, r = cv2.split(img)
    # Split HSV channels
    h, s, v = cv2.split(hsv)

    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    fig.suptitle(f'Exercise 1.4 — cv2.split() Channels: {name.replace("_"," ").title()}',
                 fontsize=12, fontweight='bold')

    # Top row: original + BGR channels
    axes[0, 0].imshow(rgb(img));      axes[0, 0].set_title('Original');     axes[0, 0].axis('off')
    axes[0, 1].imshow(r, cmap='Reds');   axes[0, 1].set_title('R channel'); axes[0, 1].axis('off')
    axes[0, 2].imshow(g, cmap='Greens'); axes[0, 2].set_title('G channel'); axes[0, 2].axis('off')
    axes[0, 3].imshow(b, cmap='Blues');  axes[0, 3].set_title('B channel'); axes[0, 3].axis('off')

    # Bottom row: HSV image + H, S, V channels
    axes[1, 0].imshow(hsv);                   axes[1, 0].set_title('HSV image');   axes[1, 0].axis('off')
    axes[1, 1].imshow(h, cmap='hsv');         axes[1, 1].set_title('H (Hue)');     axes[1, 1].axis('off')
    axes[1, 2].imshow(s, cmap='gray');        axes[1, 2].set_title('S (Saturation)'); axes[1, 2].axis('off')
    axes[1, 3].imshow(v, cmap='gray');        axes[1, 3].set_title('V (Value)');   axes[1, 3].axis('off')

    savefig(fig, f'ex1_4_split_channels_{name}.png')

Saved → ex1_4_split_channels_traffic_light.png
Saved → ex1_4_split_channels_person.png
Saved → ex1_4_split_channels_natural_scene.png
Saved → ex1_4_split_channels_road.png
Saved → ex1_4_split_channels_simple_object.png
Saved → ex1_4_split_channels_pepper.png
Saved → ex1_4_split_channels_fruits.png
Saved → ex1_4_split_channels_flowers.png
Saved → ex1_4_split_channels_parrot.png
Saved → ex1_4_split_channels_low_contrast.png
Saved → ex1_4_split_channels_night_scene.png


---
## Chapter 2 — Color Science and Point Operations

**Suggested Exercises:**
1. Convert the RGB image to grayscale using the **luminosity method**.
2. Create a function for **dynamic brightness adjustment** using a slider input.
3. Perform **histogram equalization** on each channel separately.


### Exercise 2.1 — Grayscale conversion using the Luminosity method

The luminosity method weights channels based on human perception:
```
Gray = 0.2126·R + 0.7152·G + 0.0722·B
```
This differs from the simple average (0.333·R + 0.333·G + 0.333·B) and from OpenCV's default (which uses slightly different weights).


In [6]:
def luminosity_gray(img_bgr):
    """Convert BGR image to grayscale using the ITU-R BT.709 luminosity formula."""
    img_float = img_bgr.astype(np.float32)
    b, g, r   = img_float[:,:,0], img_float[:,:,1], img_float[:,:,2]
    gray      = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return np.clip(gray, 0, 255).astype(np.uint8)

for name in IMAGE_NAMES:
    img = shrink(load(name))
    gray_lum = luminosity_gray(img)
    gray_avg = np.mean(img.astype(np.float32), axis=2).astype(np.uint8)
    gray_cv2 = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f'Exercise 2.1 — Grayscale Methods: {name.replace("_"," ").title()}',
                 fontsize=11, fontweight='bold')
    axes[0].imshow(rgb(img));                    axes[0].set_title('Original');            axes[0].axis('off')
    axes[1].imshow(gray_lum,  cmap='gray');      axes[1].set_title('Luminosity Method');   axes[1].axis('off')
    axes[2].imshow(gray_avg,  cmap='gray');      axes[2].set_title('Simple Average');      axes[2].axis('off')
    axes[3].imshow(gray_cv2,  cmap='gray');      axes[3].set_title('OpenCV Default');      axes[3].axis('off')
    savefig(fig, f'ex2_1_luminosity_gray_{name}.png')

Saved → ex2_1_luminosity_gray_traffic_light.png
Saved → ex2_1_luminosity_gray_person.png
Saved → ex2_1_luminosity_gray_natural_scene.png
Saved → ex2_1_luminosity_gray_road.png
Saved → ex2_1_luminosity_gray_simple_object.png
Saved → ex2_1_luminosity_gray_pepper.png
Saved → ex2_1_luminosity_gray_fruits.png
Saved → ex2_1_luminosity_gray_flowers.png
Saved → ex2_1_luminosity_gray_parrot.png
Saved → ex2_1_luminosity_gray_low_contrast.png
Saved → ex2_1_luminosity_gray_night_scene.png


### Exercise 2.2 — Dynamic brightness adjustment function

We implement a `adjust_brightness(img, delta)` function and demonstrate it across a range of delta values (−100 to +100) to simulate a slider.


In [7]:
def adjust_brightness(img, delta):
    """
    Shift all pixel values by delta.
    delta > 0 brightens; delta < 0 darkens.
    Values are clipped to [0, 255].
    """
    return np.clip(img.astype(np.int16) + delta, 0, 255).astype(np.uint8)

# Demonstrate the slider effect on all images
deltas = [-100, -50, -25, 0, 25, 50, 100]

for name in IMAGE_NAMES:
    img = shrink(load(name))

    fig, axes = plt.subplots(1, len(deltas), figsize=(24, 4))
    fig.suptitle(f'Exercise 2.2 — Dynamic Brightness Slider: {name.replace("_"," ").title()}',
                 fontsize=11, fontweight='bold')

    for ax, d in zip(axes, deltas):
        result = adjust_brightness(img, d)
        ax.imshow(rgb(result))
        ax.set_title(f'delta={d:+d}', fontsize=8)
        ax.axis('off')

    savefig(fig, f'ex2_2_brightness_slider_{name}.png')

Saved → ex2_2_brightness_slider_traffic_light.png
Saved → ex2_2_brightness_slider_person.png
Saved → ex2_2_brightness_slider_natural_scene.png
Saved → ex2_2_brightness_slider_road.png
Saved → ex2_2_brightness_slider_simple_object.png
Saved → ex2_2_brightness_slider_pepper.png
Saved → ex2_2_brightness_slider_fruits.png
Saved → ex2_2_brightness_slider_flowers.png
Saved → ex2_2_brightness_slider_parrot.png
Saved → ex2_2_brightness_slider_low_contrast.png
Saved → ex2_2_brightness_slider_night_scene.png


### Exercise 2.3 — Histogram equalization on each RGB channel separately

Standard `cv2.equalizeHist` works on single-channel grayscale images. Here we split the image into R, G, B channels, equalize each independently, then merge and compare with the original and a naive grayscale equalization.


In [8]:
def equalize_per_channel(img_bgr):
    """Histogram-equalise each BGR channel independently, then merge."""
    channels = cv2.split(img_bgr)
    eq_channels = [cv2.equalizeHist(c) for c in channels]
    return cv2.merge(eq_channels)

def plot_histogram(ax, img_gray, title, color='steelblue'):
    hist = cv2.calcHist([img_gray], [0], None, [256], [0, 256])
    ax.fill_between(range(256), hist.ravel(), color=color, alpha=0.7)
    ax.set_title(title, fontsize=8)
    ax.set_xlim([0, 255])
    ax.set_yticks([])

for name in IMAGE_NAMES:
    img     = shrink(load(name))
    eq_img  = equalize_per_channel(img)
    gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    eq_gray = cv2.equalizeHist(gray)

    fig = plt.figure(figsize=(18, 8))
    fig.suptitle(f'Exercise 2.3 — Per-Channel Histogram Equalization: {name.replace("_"," ").title()}',
                 fontsize=12, fontweight='bold')

    # Top row: images
    ax1 = fig.add_subplot(2, 4, 1); ax1.imshow(rgb(img));    ax1.set_title('Original');         ax1.axis('off')
    ax2 = fig.add_subplot(2, 4, 2); ax2.imshow(rgb(eq_img)); ax2.set_title('Per-Channel EQ');   ax2.axis('off')
    ax3 = fig.add_subplot(2, 4, 3); ax3.imshow(gray, cmap='gray'); ax3.set_title('Grayscale');  ax3.axis('off')
    ax4 = fig.add_subplot(2, 4, 4); ax4.imshow(eq_gray, cmap='gray'); ax4.set_title('Gray EQ'); ax4.axis('off')

    # Bottom row: histograms
    ax5 = fig.add_subplot(2, 4, 5); plot_histogram(ax5, cv2.cvtColor(img,    cv2.COLOR_BGR2GRAY), 'Hist: Original')
    ax6 = fig.add_subplot(2, 4, 6); plot_histogram(ax6, cv2.cvtColor(eq_img, cv2.COLOR_BGR2GRAY), 'Hist: Per-Ch EQ', 'tomato')
    ax7 = fig.add_subplot(2, 4, 7); plot_histogram(ax7, gray,    'Hist: Grayscale')
    ax8 = fig.add_subplot(2, 4, 8); plot_histogram(ax8, eq_gray, 'Hist: Gray EQ', 'tomato')

    fig.tight_layout()
    savefig(fig, f'ex2_3_hist_eq_{name}.png')

Saved → ex2_3_hist_eq_traffic_light.png
Saved → ex2_3_hist_eq_person.png
Saved → ex2_3_hist_eq_natural_scene.png
Saved → ex2_3_hist_eq_road.png
Saved → ex2_3_hist_eq_simple_object.png
Saved → ex2_3_hist_eq_pepper.png
Saved → ex2_3_hist_eq_fruits.png
Saved → ex2_3_hist_eq_flowers.png
Saved → ex2_3_hist_eq_parrot.png
Saved → ex2_3_hist_eq_low_contrast.png
Saved → ex2_3_hist_eq_night_scene.png


---
## Chapter 3 — Image Segmentation

**Suggested Exercises:**
1. Apply **Otsu's thresholding** to different test images.
2. **Compare Sobel, Prewitt, and Canny** edge detectors.
3. Create a **region-growing algorithm** with user-defined seed input.


### Exercise 3.1 — Otsu's thresholding on all test images

In [9]:
for name in IMAGE_NAMES:
    img  = shrink(load(name))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    thresh_val, bw_otsu = cv2.threshold(gray, 0, 255,
                                        cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(
        f'Exercise 3.1 — Otsu Thresholding: {name.replace("_"," ").title()}  '
        f'(T = {thresh_val:.0f})',
        fontsize=11, fontweight='bold')
    axes[0].imshow(rgb(img));           axes[0].set_title('Original');          axes[0].axis('off')
    axes[1].imshow(gray, cmap='gray');  axes[1].set_title('Grayscale');         axes[1].axis('off')
    axes[2].imshow(bw_otsu, cmap='gray'); axes[2].set_title(f"Otsu (T={thresh_val:.0f})"); axes[2].axis('off')
    savefig(fig, f'ex3_1_otsu_{name}.png')

Saved → ex3_1_otsu_traffic_light.png
Saved → ex3_1_otsu_person.png
Saved → ex3_1_otsu_natural_scene.png
Saved → ex3_1_otsu_road.png
Saved → ex3_1_otsu_simple_object.png
Saved → ex3_1_otsu_pepper.png
Saved → ex3_1_otsu_fruits.png
Saved → ex3_1_otsu_flowers.png
Saved → ex3_1_otsu_parrot.png
Saved → ex3_1_otsu_low_contrast.png
Saved → ex3_1_otsu_night_scene.png


### Exercise 3.2 — Compare Sobel, Prewitt, and Canny edge detectors

All three detectors find edges but differ in approach:
- **Sobel**: 3×3 gradient kernel with central-difference weighting
- **Prewitt**: Similar to Sobel but with equal weights (no central emphasis)
- **Canny**: Multi-stage pipeline — Gaussian smoothing → gradient → NMS → hysteresis


In [10]:
def sobel_edges(gray):
    gx  = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy  = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    return np.clip(mag / mag.max() * 255, 0, 255).astype(np.uint8)

def prewitt_edges(gray):
    kx = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
    ky = np.array([[-1,-1,-1], [ 0, 0, 0], [ 1, 1, 1]], dtype=np.float32)
    gx  = cv2.filter2D(gray.astype(np.float32), -1, kx)
    gy  = cv2.filter2D(gray.astype(np.float32), -1, ky)
    mag = np.sqrt(gx**2 + gy**2)
    return np.clip(mag / mag.max() * 255, 0, 255).astype(np.uint8)

def canny_edges(gray):
    return cv2.Canny(gray, 50, 150)

for name in IMAGE_NAMES:
    img  = shrink(load(name))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    sobel   = sobel_edges(gray)
    prewitt = prewitt_edges(gray)
    canny   = canny_edges(gray)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f'Exercise 3.2 — Edge Detector Comparison: {name.replace("_"," ").title()}',
                 fontsize=11, fontweight='bold')
    axes[0].imshow(rgb(img));             axes[0].set_title('Original');  axes[0].axis('off')
    axes[1].imshow(sobel,   cmap='gray'); axes[1].set_title('Sobel');     axes[1].axis('off')
    axes[2].imshow(prewitt, cmap='gray'); axes[2].set_title('Prewitt');   axes[2].axis('off')
    axes[3].imshow(canny,   cmap='gray'); axes[3].set_title('Canny');     axes[3].axis('off')
    savefig(fig, f'ex3_2_edge_compare_{name}.png')

Saved → ex3_2_edge_compare_traffic_light.png
Saved → ex3_2_edge_compare_person.png
Saved → ex3_2_edge_compare_natural_scene.png
Saved → ex3_2_edge_compare_road.png
Saved → ex3_2_edge_compare_simple_object.png
Saved → ex3_2_edge_compare_pepper.png
Saved → ex3_2_edge_compare_fruits.png
Saved → ex3_2_edge_compare_flowers.png
Saved → ex3_2_edge_compare_parrot.png
Saved → ex3_2_edge_compare_low_contrast.png
Saved → ex3_2_edge_compare_night_scene.png


### Exercise 3.3 — Region-growing algorithm with user-defined seed

Region growing starts from a seed pixel and expands to neighbours whose intensity is within a tolerance `thresh` of the seed value. We implement it with a BFS queue and apply it to selected images.


In [11]:
def region_grow(gray, seed_rc, thresh=15):
    """
    Grow a region from seed (row, col) in a grayscale image.
    Neighbours within abs(intensity - seed_value) <= thresh are added.
    Returns a binary mask of the grown region.
    """
    h, w      = gray.shape
    visited   = np.zeros((h, w), dtype=bool)
    mask      = np.zeros((h, w), dtype=np.uint8)
    seed_val  = int(gray[seed_rc])
    queue     = [seed_rc]
    visited[seed_rc] = True

    while queue:
        r, c = queue.pop(0)
        if abs(int(gray[r, c]) - seed_val) <= thresh:
            mask[r, c] = 255
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc]:
                    visited[nr, nc] = True
                    queue.append((nr, nc))
    return mask

# Define a representative seed for each image (row, col as fraction of image size)
# These are chosen to land on an interesting region in each image
seed_fractions = {
    'traffic_light':  (0.12, 0.35),   # red light area
    'person':         (0.35, 0.50),   # face area
    'natural_scene':  (0.80, 0.25),   # foreground hillside
    'road':           (0.60, 0.50),   # road surface
    'simple_object':  (0.50, 0.50),   # centre of bottle
    'pepper':         (0.45, 0.20),   # yellow pepper
    'fruits':         (0.70, 0.20),   # orange foreground
    'flowers':        (0.35, 0.08),   # orange flower
    'parrot':         (0.30, 0.10),   # blue feather area
    'low_contrast':   (0.50, 0.50),   # centre
    'night_scene':    (0.80, 0.30),   # road surface
}

for name in IMAGE_NAMES:
    img   = shrink(load(name), max_side=400)
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w  = gray.shape
    fr, fc = seed_fractions[name]
    seed  = (int(fr * h), int(fc * w))

    grown = region_grow(gray, seed, thresh=20)

    # Overlay the seed point and grown region on original
    overlay = rgb(img).copy()
    overlay[grown > 0] = (overlay[grown > 0] * 0.4 + np.array([255, 80, 0]) * 0.6).astype(np.uint8)
    cv2.circle(overlay, (seed[1], seed[0]), 5, (255, 255, 0), -1)   # yellow dot = seed

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(
        f'Exercise 3.3 — Region Growing: {name.replace("_"," ").title()}  '
        f'(seed={seed}, thresh=20)',
        fontsize=10, fontweight='bold')
    axes[0].imshow(rgb(img));          axes[0].set_title('Original');      axes[0].axis('off')
    axes[1].imshow(grown, cmap='gray'); axes[1].set_title('Grown Region'); axes[1].axis('off')
    axes[2].imshow(overlay);           axes[2].set_title('Overlay');       axes[2].axis('off')
    savefig(fig, f'ex3_3_region_grow_{name}.png')

Saved → ex3_3_region_grow_traffic_light.png
Saved → ex3_3_region_grow_person.png
Saved → ex3_3_region_grow_natural_scene.png
Saved → ex3_3_region_grow_road.png
Saved → ex3_3_region_grow_simple_object.png
Saved → ex3_3_region_grow_pepper.png
Saved → ex3_3_region_grow_fruits.png
Saved → ex3_3_region_grow_flowers.png
Saved → ex3_3_region_grow_parrot.png
Saved → ex3_3_region_grow_low_contrast.png
Saved → ex3_3_region_grow_night_scene.png


---
## Chapter 4 — Morphological Image Processing

**Suggested Exercises:**
1. Use **different structuring elements** (disk, diamond) and compare outputs.
2. Apply **opening and closing to a noisy image** and evaluate effectiveness.
3. Use morphological operations to **extract text** from an image.


### Exercise 4.1 — Compare structuring element shapes

We test four structuring element shapes on every image:
**Square**, **Ellipse (disk)**, **Cross**, and **Diamond** (custom kernel).


In [14]:
def make_diamond(size=5):
    """Create a diamond-shaped structuring element of given size."""
    se = np.zeros((size, size), dtype=np.uint8)
    mid = size // 2
    for r in range(size):
        for c in range(size):
            if abs(r - mid) + abs(c - mid) <= mid:
                se[r, c] = 1
    return se

se_square  = cv2.getStructuringElement(cv2.MORPH_RECT,    (7, 7))
se_disk    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
se_cross   = cv2.getStructuringElement(cv2.MORPH_CROSS,   (7, 7))
se_diamond = make_diamond(7)

se_shapes = {
    'Square (7x7)':  se_square,
    'Disk (7x7)':    se_disk,
    'Cross (7x7)':   se_cross,
    'Diamond (7x7)': se_diamond,
}

for name in IMAGE_NAMES:
    img   = shrink(load(name))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    n_se = len(se_shapes)
    fig, axes = plt.subplots(2, n_se + 1, figsize=(22, 9))
    fig.suptitle(
        f'Exercise 4.1 — SE Shape Comparison: {name.replace("_"," ").title()}',
        fontsize=12,
        fontweight='bold'
    )

    # Column 0: original binary
    axes[0, 0].imshow(rgb(img))
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')

    axes[1, 0].imshow(bw, cmap='gray')
    axes[1, 0].set_title('Binary')
    axes[1, 0].axis('off')

    for col, (label, se) in enumerate(se_shapes.items(), start=1):
        eroded  = cv2.erode(bw, se)
        dilated = cv2.dilate(bw, se)

        axes[0, col].imshow(eroded, cmap='gray')
        axes[0, col].set_title(f'Erode\n{label}', fontsize=8)
        axes[0, col].axis('off')

        axes[1, col].imshow(dilated, cmap='gray')
        axes[1, col].set_title(f'Dilate\n{label}', fontsize=8)
        axes[1, col].axis('off')

    savefig(fig, f'ex4_1_se_shapes_{name}.png')

Saved → ex4_1_se_shapes_traffic_light.png
Saved → ex4_1_se_shapes_person.png
Saved → ex4_1_se_shapes_natural_scene.png
Saved → ex4_1_se_shapes_road.png
Saved → ex4_1_se_shapes_simple_object.png
Saved → ex4_1_se_shapes_pepper.png
Saved → ex4_1_se_shapes_fruits.png
Saved → ex4_1_se_shapes_flowers.png
Saved → ex4_1_se_shapes_parrot.png
Saved → ex4_1_se_shapes_low_contrast.png
Saved → ex4_1_se_shapes_night_scene.png


### Exercise 4.2 — Opening and closing on noisy images

We add **salt-and-pepper noise** to each image, then apply:
- **Opening** (erode → dilate): removes salt noise (white specks)
- **Closing** (dilate → erode): fills pepper noise (black holes)
- **Opening + Closing**: handles both types together


In [15]:
def add_sp_noise(img_gray, amount=0.05, seed=42):
    """Add salt-and-pepper noise to a grayscale image."""
    noisy = img_gray.copy()
    rng = np.random.default_rng(seed)
    noisy[rng.random(img_gray.shape) < amount / 2] = 255  # salt
    noisy[rng.random(img_gray.shape) < amount / 2] = 0    # pepper
    return noisy

se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

for name in IMAGE_NAMES:
    img   = shrink(load(name))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    noisy   = add_sp_noise(bw, amount=0.08)
    opened  = cv2.morphologyEx(noisy, cv2.MORPH_OPEN,  se)
    closed  = cv2.morphologyEx(noisy, cv2.MORPH_CLOSE, se)
    both    = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, se)

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(f'Exercise 4.2 — Noise Removal with Opening/Closing: {name.replace("_"," ").title()}',
                 fontsize=11, fontweight='bold')
    for ax, im, title in zip(axes,
            [bw, noisy, opened, closed, both],
            ['Clean Binary', 'Noisy (+S&P 8%)', 'Opening', 'Closing', 'Open + Close']):
        ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')
    savefig(fig, f'ex4_2_open_close_noise_{name}.png')

Saved → ex4_2_open_close_noise_traffic_light.png
Saved → ex4_2_open_close_noise_person.png
Saved → ex4_2_open_close_noise_natural_scene.png
Saved → ex4_2_open_close_noise_road.png
Saved → ex4_2_open_close_noise_simple_object.png
Saved → ex4_2_open_close_noise_pepper.png
Saved → ex4_2_open_close_noise_fruits.png
Saved → ex4_2_open_close_noise_flowers.png
Saved → ex4_2_open_close_noise_parrot.png
Saved → ex4_2_open_close_noise_low_contrast.png
Saved → ex4_2_open_close_noise_night_scene.png


### Exercise 4.3 — Use morphological operations to extract/enhance text-like features

Morphological operations can isolate fine detail (text, thin lines) using top-hat and black-hat transforms. We demonstrate this on the images that contain structured or text-like patterns (road markings, bottle label, traffic sign).


In [16]:
# Use a horizontal structuring element to isolate horizontal features
# and a vertical one to isolate vertical features — common in text extraction pipelines

se_h = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))   # horizontal bar
se_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25))   # vertical bar
se_sq = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

for name in IMAGE_NAMES:
    img   = shrink(load(name))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    tophat_h   = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT,   se_h)
    tophat_v   = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT,   se_v)
    blackhat   = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, se_sq)
    gradient   = cv2.morphologyEx(gray, cv2.MORPH_GRADIENT, se_sq)

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(f'Exercise 4.3 — Morphological Feature Extraction: {name.replace("_"," ").title()}',
                 fontsize=11, fontweight='bold')
    for ax, im, title in zip(axes,
            [gray, tophat_h, tophat_v, blackhat, gradient],
            ['Grayscale', 'Top-hat (H)', 'Top-hat (V)', 'Black-hat', 'Morph Gradient']):
        ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')
    savefig(fig, f'ex4_3_morph_features_{name}.png')

Saved → ex4_3_morph_features_traffic_light.png
Saved → ex4_3_morph_features_person.png
Saved → ex4_3_morph_features_natural_scene.png
Saved → ex4_3_morph_features_road.png
Saved → ex4_3_morph_features_simple_object.png
Saved → ex4_3_morph_features_pepper.png
Saved → ex4_3_morph_features_fruits.png
Saved → ex4_3_morph_features_flowers.png
Saved → ex4_3_morph_features_parrot.png
Saved → ex4_3_morph_features_low_contrast.png
Saved → ex4_3_morph_features_night_scene.png


---
## Chapter 5 — Image Filtering and Correlation

**Suggested Exercises:**
1. Apply filters of **different sizes (5×5, 7×7)** and compare results.
2. **Implement unsharp masking manually**.
3. **Test filtering on noisy images** and evaluate PSNR or SSIM.
4. **Correlate a small patch** against a larger image (template matching).


### Exercise 5.1 — Filter size comparison: 3×3, 5×5, 7×7

Larger kernels smooth more aggressively but lose more fine detail. We compare the box filter and Gaussian at three sizes.


In [17]:
for name in IMAGE_NAMES:
    img   = shrink(load(name))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Box filters
    box3 = cv2.blur(gray, (3, 3))
    box5 = cv2.blur(gray, (5, 5))
    box7 = cv2.blur(gray, (7, 7))

    # Gaussian filters (sigma chosen proportional to kernel size)
    g3 = cv2.GaussianBlur(gray, (3, 3), 1)
    g5 = cv2.GaussianBlur(gray, (5, 5), 2)
    g7 = cv2.GaussianBlur(gray, (7, 7), 3)

    fig, axes = plt.subplots(2, 4, figsize=(22, 9))
    fig.suptitle(f'Exercise 5.1 — Filter Size Study: {name.replace("_"," ").title()}',
                 fontsize=12, fontweight='bold')

    top_items = [('Original', gray), ('Box 3x3', box3), ('Box 5x5', box5), ('Box 7x7', box7)]
    bot_items = [('Original', gray), ('Gauss 3x3', g3), ('Gauss 5x5', g5), ('Gauss 7x7', g7)]

    for ax, (title, im) in zip(axes[0], top_items):
        ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')
    for ax, (title, im) in zip(axes[1], bot_items):
        ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')

    savefig(fig, f'ex5_1_filter_sizes_{name}.png')

Saved → ex5_1_filter_sizes_traffic_light.png
Saved → ex5_1_filter_sizes_person.png
Saved → ex5_1_filter_sizes_natural_scene.png
Saved → ex5_1_filter_sizes_road.png
Saved → ex5_1_filter_sizes_simple_object.png
Saved → ex5_1_filter_sizes_pepper.png
Saved → ex5_1_filter_sizes_fruits.png
Saved → ex5_1_filter_sizes_flowers.png
Saved → ex5_1_filter_sizes_parrot.png
Saved → ex5_1_filter_sizes_low_contrast.png
Saved → ex5_1_filter_sizes_night_scene.png


### Exercise 5.2 — Manual unsharp masking

Unsharp masking sharpens by subtracting a blurred copy:
```
sharp = original + strength × (original − blurred)
      = (1 + strength) × original − strength × blurred
```
We implement this from scratch and compare different strength values.


In [18]:
def unsharp_mask_manual(gray, sigma=2.0, strength=1.0):
    """
    Manual unsharp masking.
    1. Blur the image with a Gaussian.
    2. Compute the detail mask: mask = original - blurred
    3. Add scaled mask back: sharpened = original + strength * mask
    """
    blurred = cv2.GaussianBlur(gray.astype(np.float32), (0, 0), sigma)
    mask    = gray.astype(np.float32) - blurred
    sharp   = gray.astype(np.float32) + strength * mask
    return np.clip(sharp, 0, 255).astype(np.uint8)

for name in IMAGE_NAMES:
    img   = shrink(load(name))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    strengths = [0.5, 1.0, 1.5, 2.0, 3.0]
    results   = [unsharp_mask_manual(gray, sigma=2, strength=s) for s in strengths]

    fig, axes = plt.subplots(1, len(strengths) + 1, figsize=(24, 4))
    fig.suptitle(f'Exercise 5.2 — Manual Unsharp Masking: {name.replace("_"," ").title()}',
                 fontsize=11, fontweight='bold')
    axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Original', fontsize=9); axes[0].axis('off')
    for ax, im, s in zip(axes[1:], results, strengths):
        ax.imshow(im, cmap='gray'); ax.set_title(f'strength={s}', fontsize=9); ax.axis('off')
    savefig(fig, f'ex5_2_unsharp_mask_{name}.png')

Saved → ex5_2_unsharp_mask_traffic_light.png
Saved → ex5_2_unsharp_mask_person.png
Saved → ex5_2_unsharp_mask_natural_scene.png
Saved → ex5_2_unsharp_mask_road.png
Saved → ex5_2_unsharp_mask_simple_object.png
Saved → ex5_2_unsharp_mask_pepper.png
Saved → ex5_2_unsharp_mask_fruits.png
Saved → ex5_2_unsharp_mask_flowers.png
Saved → ex5_2_unsharp_mask_parrot.png
Saved → ex5_2_unsharp_mask_low_contrast.png
Saved → ex5_2_unsharp_mask_night_scene.png


### Exercise 5.3 — Filter noisy images and evaluate PSNR and SSIM

We add salt-and-pepper noise to each image, apply three filters, then measure how well each filter restores the original using:
- **PSNR** (Peak Signal-to-Noise Ratio) — higher is better (dB)
- **SSIM** (Structural Similarity Index) — closer to 1.0 is better


In [19]:
results_table = []

for name in IMAGE_NAMES:
    img   = shrink(load(name))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

    # Add salt-and-pepper noise
    rng   = np.random.default_rng(0)
    noisy = gray.copy()
    noisy[rng.random(gray.shape) < 0.05] = 1.0
    noisy[rng.random(gray.shape) < 0.05] = 0.0

    # Convert noisy to uint8 for median filter (requires uint8)
    noisy_u8  = (noisy * 255).astype(np.uint8)
    gray_u8   = (gray  * 255).astype(np.uint8)

    # Apply filters
    box_f  = cv2.blur(noisy_u8, (3, 3)).astype(np.float32) / 255.0
    gauss_f= cv2.GaussianBlur(noisy_u8, (5, 5), 1).astype(np.float32) / 255.0
    med_f  = cv2.medianBlur(noisy_u8, 5).astype(np.float32) / 255.0

    data_range = 1.0
    scores = {}
    for label, filt in [('Box 3x3', box_f), ('Gauss 5x5', gauss_f), ('Median 5x5', med_f)]:
        p = psnr(gray, filt, data_range=data_range)
        s = ssim(gray, filt, data_range=data_range)
        scores[label] = (p, s)

    # also score the raw noisy image
    psnr_n = psnr(gray, noisy, data_range=data_range)
    ssim_n = ssim(gray, noisy, data_range=data_range)

    results_table.append({
        'image': name,
        'noisy_psnr': psnr_n, 'noisy_ssim': ssim_n,
        **{f'{k}_psnr': v[0] for k, v in scores.items()},
        **{f'{k}_ssim': v[1] for k, v in scores.items()},
    })

    # Visual comparison
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(
        f'Exercise 5.3 — Filtering + Metrics: {name.replace("_"," ").title()}',
        fontsize=11, fontweight='bold')
    items = [
        (gray,   f'Original'),
        (noisy,  f'Noisy\nPSNR={psnr_n:.1f} SSIM={ssim_n:.3f}'),
        (box_f,  f'Box 3x3\nPSNR={scores["Box 3x3"][0]:.1f} SSIM={scores["Box 3x3"][1]:.3f}'),
        (gauss_f,f'Gauss 5x5\nPSNR={scores["Gauss 5x5"][0]:.1f} SSIM={scores["Gauss 5x5"][1]:.3f}'),
        (med_f,  f'Median 5x5\nPSNR={scores["Median 5x5"][0]:.1f} SSIM={scores["Median 5x5"][1]:.3f}'),
    ]
    for ax, (im, title) in zip(axes, items):
        ax.imshow(im, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title, fontsize=8)
        ax.axis('off')
    savefig(fig, f'ex5_3_psnr_ssim_{name}.png')

# Print summary table
print(f'\n{"Image":<20} {"Noisy PSNR":>11} {"Box PSNR":>10} {"Gauss PSNR":>11} {"Median PSNR":>12}')
print('-' * 68)
for row in results_table:
    print(f'{row["image"]:<20} {row["noisy_psnr"]:>11.2f} '
          f'{row["Box 3x3_psnr"]:>10.2f} '
          f'{row["Gauss 5x5_psnr"]:>11.2f} '
          f'{row["Median 5x5_psnr"]:>12.2f}')

Saved → ex5_3_psnr_ssim_traffic_light.png
Saved → ex5_3_psnr_ssim_person.png
Saved → ex5_3_psnr_ssim_natural_scene.png
Saved → ex5_3_psnr_ssim_road.png
Saved → ex5_3_psnr_ssim_simple_object.png
Saved → ex5_3_psnr_ssim_pepper.png
Saved → ex5_3_psnr_ssim_fruits.png
Saved → ex5_3_psnr_ssim_flowers.png
Saved → ex5_3_psnr_ssim_parrot.png
Saved → ex5_3_psnr_ssim_low_contrast.png
Saved → ex5_3_psnr_ssim_night_scene.png

Image                 Noisy PSNR   Box PSNR  Gauss PSNR  Median PSNR
--------------------------------------------------------------------
traffic_light              14.83      23.27       24.19        33.01
person                     14.84      23.48       24.45        38.09
natural_scene              15.18      22.48       23.24        25.84
road                       15.29      22.48       23.16        24.97
simple_object              13.84      22.12       22.99        41.05
pepper                     13.88      22.07       22.88        35.91
fruits                     13.9

### Exercise 5.4 — Correlate a small patch against a larger image (template matching)

We cut a small patch from each image and use `cv2.matchTemplate` (which performs normalised cross-correlation) to locate it back in the full image. The brightest point in the correlation map is the best match.


In [20]:
for name in IMAGE_NAMES:
    img   = shrink(load(name), max_side=500)
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w  = gray.shape

    # Cut a 40x40 patch from the image centre
    pr, pc = h // 2, w // 2
    patch  = gray[pr-20:pr+20, pc-20:pc+20]

    # Normalised cross-correlation map
    corr_map = cv2.matchTemplate(gray, patch, cv2.TM_CCOEFF_NORMED)

    # Find location of best match
    _, max_val, _, max_loc = cv2.minMaxLoc(corr_map)
    match_top_left     = max_loc
    match_bottom_right = (max_loc[0] + 40, max_loc[1] + 40)

    # Draw the matched rectangle on a colour copy
    vis = rgb(img).copy()
    cv2.rectangle(vis, match_top_left, match_bottom_right, (255, 50, 50), 3)
    cv2.rectangle(vis, (pc-20, pr-20), (pc+20, pr+20), (50, 255, 50), 2)

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(
        f'Exercise 5.4 — Template Matching: {name.replace("_"," ").title()}  '
        f'(match score={max_val:.3f})',
        fontsize=11, fontweight='bold')
    axes[0].imshow(rgb(img));           axes[0].set_title('Original');         axes[0].axis('off')
    axes[1].imshow(patch, cmap='gray'); axes[1].set_title('Patch (template)'); axes[1].axis('off')
    axes[2].imshow(corr_map, cmap='hot'); axes[2].set_title('Correlation Map'); axes[2].axis('off')
    axes[3].imshow(vis);                axes[3].set_title('Match (red=found, green=source)'); axes[3].axis('off')
    savefig(fig, f'ex5_4_template_match_{name}.png')

Saved → ex5_4_template_match_traffic_light.png
Saved → ex5_4_template_match_person.png
Saved → ex5_4_template_match_natural_scene.png
Saved → ex5_4_template_match_road.png
Saved → ex5_4_template_match_simple_object.png
Saved → ex5_4_template_match_pepper.png
Saved → ex5_4_template_match_fruits.png
Saved → ex5_4_template_match_flowers.png
Saved → ex5_4_template_match_parrot.png
Saved → ex5_4_template_match_low_contrast.png
Saved → ex5_4_template_match_night_scene.png


---
## Done
All chapter exercises completed. Check the output directory for all saved figures.

In [21]:
import glob
output_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.png')))
print(f'Total output files: {len(output_files)}')
for f in output_files:
    print(' ', os.path.basename(f))

Total output files: 166
  ex1_1_load_display_all.png
  ex1_3_colorspaces_flowers.png
  ex1_3_colorspaces_fruits.png
  ex1_3_colorspaces_low_contrast.png
  ex1_3_colorspaces_natural_scene.png
  ex1_3_colorspaces_night_scene.png
  ex1_3_colorspaces_parrot.png
  ex1_3_colorspaces_pepper.png
  ex1_3_colorspaces_person.png
  ex1_3_colorspaces_road.png
  ex1_3_colorspaces_simple_object.png
  ex1_3_colorspaces_traffic_light.png
  ex1_4_split_channels_flowers.png
  ex1_4_split_channels_fruits.png
  ex1_4_split_channels_low_contrast.png
  ex1_4_split_channels_natural_scene.png
  ex1_4_split_channels_night_scene.png
  ex1_4_split_channels_parrot.png
  ex1_4_split_channels_pepper.png
  ex1_4_split_channels_person.png
  ex1_4_split_channels_road.png
  ex1_4_split_channels_simple_object.png
  ex1_4_split_channels_traffic_light.png
  ex2_1_luminosity_gray_flowers.png
  ex2_1_luminosity_gray_fruits.png
  ex2_1_luminosity_gray_low_contrast.png
  ex2_1_luminosity_gray_natural_scene.png
  ex2_1_luminosi